In [4]:
import torch
from PIL import Image
import numpy as np
from transformers import AutoModel, CLIPVisionModel, AutoImageProcessor
import plotly.graph_objects as go
from pathlib import Path
import re
import pandas as pd
from plotly.subplots import make_subplots

def extract_features(model, processor, image):
    """提取图像特征"""
    inputs = processor(images=image, return_tensors="pt")
    
    with torch.no_grad():
        outputs = model(**inputs)
        features = outputs.last_hidden_state[:, 0, :]
        features = torch.nn.functional.normalize(features, p=2, dim=1)
    
    return features

def parse_image_name(filename):
    """解析图像文件名，提取参数值"""
    pattern = r'(\w+)_illumination(\d+)_placement(\d+)_orientation(\d+)_angle(\d+)\.jpg'
    match = re.match(pattern, filename)
    if match:
        return {
            'category': match.group(1),
            'illumination': int(match.group(2)),
            'placement': int(match.group(3)),
            'orientation': int(match.group(4)),
            'angle': int(match.group(5))
        }
    return None

def create_similarity_heatmap(features_dict, metadata_df, condition1, condition2, title, categories):
    """创建多类别的相似度热力图"""
    # 为每个类别创建子图
    fig = make_subplots(
        rows=4, cols=3,
        subplot_titles=categories,
        vertical_spacing=0.08,
        horizontal_spacing=0.05
    )
    
    # 获取所有可能的条件值
    unique_vals1 = sorted(metadata_df[condition1].unique())
    unique_vals2 = sorted(metadata_df[condition2].unique())
    
    # 记录全局最大最小值用于统一颜色范围
    global_min = float('inf')
    global_max = float('-inf')
    
    # 先计算所有相似度矩阵
    similarity_matrices = {}
    for idx, category in enumerate(categories):
        cat_mask = metadata_df['category'] == category
        if not cat_mask.any():
            continue
            
        features = features_dict[category]
        cat_metadata = metadata_df[cat_mask]
        
        sim_matrix = np.zeros((len(unique_vals1), len(unique_vals2)))
        
        for i, val1 in enumerate(unique_vals1):
            indices1 = cat_metadata[condition1] == val1
            for j, val2 in enumerate(unique_vals2):
                indices2 = cat_metadata[condition2] == val2
                
                feat1 = features[indices1]
                feat2 = features[indices2]
                
                if len(feat1) > 0 and len(feat2) > 0:
                    similarity = torch.nn.functional.cosine_similarity(
                        feat1.unsqueeze(1),
                        feat2.unsqueeze(0),
                        dim=2
                    )
                    if condition1 == condition2:
                        mask = ~torch.eye(similarity.shape[0], dtype=bool)
                        sim_matrix[i, j] = similarity[mask].mean().item()
                    else:
                        sim_matrix[i, j] = similarity.mean().item()
        
        similarity_matrices[category] = sim_matrix
        global_min = min(global_min, sim_matrix.min())
        global_max = max(global_max, sim_matrix.max())
    
    # 添加热力图
    for idx, category in enumerate(categories):
        if category not in similarity_matrices:
            continue
            
        row = (idx // 3) + 1
        col = (idx % 3) + 1
        
        fig.add_trace(
            go.Heatmap(
                z=similarity_matrices[category],
                x=[str(v) for v in unique_vals2],
                y=[str(v) for v in unique_vals1],
                colorscale='RdBu_r',
                zmid=(global_max + global_min) / 2,
                zmin=global_min,
                zmax=global_max,
                text=np.around(similarity_matrices[category], decimals=3),
                texttemplate='%{text}',
                textfont={"size": 8},
                hoverongaps=False,
                hovertemplate=f"{category}<br>{condition1}: %{{y}}<br>{condition2}: %{{x}}<br>Similarity: %{{z:.3f}}<extra></extra>"
            ),
            row=row, col=col
        )
        
        # 更新轴标签
        fig.update_xaxes(title_text=condition2 if row == 4 else "", row=row, col=col)
        fig.update_yaxes(title_text=condition1 if col == 1 else "", row=row, col=col)
    
    fig.update_layout(
        title=title,
        height=1200,
        width=1200,
        showlegend=False,
    )
    
    return fig

def analyze_conditions():
    print("Loading models...")
    dino_model = AutoModel.from_pretrained('Leonardo6/dino-datacomp-12m-new')
    clip_model = CLIPVisionModel.from_pretrained('Leonardo6/clip-datacomp-12m-16')
    
    dino_processor = AutoImageProcessor.from_pretrained('Leonardo6/dino-datacomp-12m-new')
    clip_processor = AutoImageProcessor.from_pretrained('Leonardo6/clip-datacomp-12m-16')
    
    # 所有类别
    categories = ['tape', 'brush', 'cup', 'tomato', 'basket', 'yarn', 
                 'dolphin', 'whale', 'plant', 'greenapple']
    
    base_path = Path('/pasteur2/u/yuhuiz/yiming/experiments/dataset/DAISO-10')
    
    # 存储特征和元数据
    features_dino = {cat: [] for cat in categories}
    features_clip = {cat: [] for cat in categories}
    metadata = []
    
    print("Processing images...")
    for category in categories:
        category_path = base_path / category
        image_files = sorted(list(category_path.glob('*.jpg')))
        
        for img_path in image_files:
            params = parse_image_name(img_path.name)
            if params:
                img = Image.open(img_path)
                dino_feat = extract_features(dino_model, dino_processor, img)
                clip_feat = extract_features(clip_model, clip_processor, img)
                
                features_dino[category].append(dino_feat)
                features_clip[category].append(clip_feat)
                metadata.append(params)
    
    # 合并特征并转换为tensor
    for category in categories:
        if features_dino[category]:  # 检查是否有特征
            features_dino[category] = torch.cat(features_dino[category])
            features_clip[category] = torch.cat(features_clip[category])
    
    metadata_df = pd.DataFrame(metadata)
    
    # 分析所有条件对的相关性
    conditions = ['angle', 'orientation', 'placement', 'illumination']
    
    for i, cond1 in enumerate(conditions):
        for j, cond2 in enumerate(conditions[i:], i):
            # DINO热力图
            dino_fig = create_similarity_heatmap(
                features_dino,
                metadata_df,
                cond1,
                cond2,
                f'DINO Similarity: {cond1} vs {cond2}',
                categories
            )
            dino_fig.show()
            
            # CLIP热力图
            clip_fig = create_similarity_heatmap(
                features_clip,
                metadata_df,
                cond1,
                cond2,
                f'CLIP Similarity: {cond1} vs {cond2}',
                categories
            )
            clip_fig.show()
            
    # 计算每个类别在不同条件下的平均敏感度
    print("\nAverage sensitivity analysis:")
    for condition in conditions:
        print(f"\nCondition: {condition}")
        print("Category      DINO    CLIP")
        print("-" * 30)
        
        for category in categories:
            if category not in features_dino or len(features_dino[category]) == 0:
                continue
                
            cat_mask = metadata_df['category'] == category
            cat_metadata = metadata_df[cat_mask]
            
            # 计算DINO的平均敏感度
            dino_sensitivities = []
            clip_sensitivities = []
            
            unique_vals = sorted(cat_metadata[condition].unique())
            for val in unique_vals:
                val_mask = cat_metadata[condition] == val
                dino_feats = features_dino[category][val_mask]
                clip_feats = features_clip[category][val_mask]
                
                if len(dino_feats) > 1:
                    dino_sim = torch.nn.functional.cosine_similarity(
                        dino_feats.unsqueeze(1),
                        dino_feats.unsqueeze(0),
                        dim=2
                    )
                    clip_sim = torch.nn.functional.cosine_similarity(
                        clip_feats.unsqueeze(1),
                        clip_feats.unsqueeze(0),
                        dim=2
                    )
                    
                    mask = ~torch.eye(dino_sim.shape[0], dtype=bool)
                    dino_sensitivities.append(dino_sim[mask].mean().item())
                    clip_sensitivities.append(clip_sim[mask].mean().item())
            
            if dino_sensitivities:
                dino_avg = np.mean(dino_sensitivities)
                clip_avg = np.mean(clip_sensitivities)
                print(f"{category:12} {dino_avg:.3f}  {clip_avg:.3f}")

if __name__ == "__main__":
    analyze_conditions()

Loading models...
Processing images...


KeyError: 0